In [ ]:
import csv
import shutil
import os
import cv2
import numpy as np
import yaml, random
from PIL import Image
from glob import glob
from matplotlib import pyplot as plt
from torchvision import transforms as T
from ultralytics import YOLO
from pathlib import Path

In [ ]:
from pathlib import Path
import shutil
import os

# Define project paths for better organization and maintainability

PROJECT_ROOT = Path.cwd()             
DATA_ROOT = PROJECT_ROOT / "data"      # local dataset folder
RUNS_ROOT = PROJECT_ROOT / "runs"

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Runs root:", RUNS_ROOT)

assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}"

In [ ]:
%%writefile data.yaml
path: .
train: data/train/images
val: data/val/images
test: data/test/images

names:
  0: smoke
  1: fire

In [ ]:
import random 
from matplotlib import pyplot as plt
import cv2

# Visualize some images with their bounding boxes from the training set

train_img_path = DATA_ROOT / "train/images"
train_label_path = DATA_ROOT / "train/labels"

sample_imgs = random.sample(os.listdir(train_img_path), 9)

def plot_image_with_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    label_file = label_path.replace('.jpg', '.txt').replace('.png', '.txt')
    if os.path.exists(label_file):
        with open(label_file, "r") as f:
            for line in f.readlines():
                cls, x_center, y_center, bw, bh = map(float, line.strip().split())
                x1 = int((x_center - bw/2) * w)
                y1 = int((y_center - bh/2) * h)
                x2 = int((x_center + bw/2) * w)
                y2 = int((y_center + bh/2) * h)
                color = (0, 255, 0) if cls == 0 else (0, 0, 255)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cls_name = "smoke" if cls == 0 else "fire"
                cv2.putText(img, cls_name, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

plt.figure(figsize=(18, 12))
for i, img_name in enumerate(sample_imgs):
    img_path = os.path.join(train_img_path, img_name)
    label_path = os.path.join(train_label_path, img_name)
    plt.subplot(3, 3, i+1)
    plt.imshow(plot_image_with_boxes(img_path, label_path))
    plt.axis('off')
    plt.title(img_name)
plt.tight_layout()
plt.show()


In [ ]:
# Checking for paired images and labels to ensure data integrity before training and
# making sure there are no missing labels or images that could cause issues during training.

missing_labels = []
missing_images = []

image_bases = {os.path.splitext(f)[0] for f in os.listdir(train_img_path)}
label_bases = {os.path.splitext(f)[0] for f in os.listdir(train_label_path)}

for base in image_bases:
    if base not in label_bases:
        missing_labels.append(base)

for base in label_bases:
    if base not in image_bases:
        missing_images.append(base)

print("Images without labels:", len(missing_labels))
print("Labels without images:", len(missing_images))



In [ ]:
# Cont. data check

missing_images = label_bases - image_bases
missing_labels = image_bases - label_bases

print("Images:", len(image_bases))
print("Labels:", len(label_bases))

print("Missing images:", len(missing_images))
print("Missing labels:", len(missing_labels))

print("\nSample missing images:", list(missing_images)[:10])
print("\nSample missing labels:", list(missing_labels)[:10])

In [ ]:
## this function checks the format of the YOLO labels to ensure they are correct before training, 
# which can help prevent training errors and improve model performance

def validate_yolo_labels(label_dir, valid_classes=[0, 1], verbose=True):
    bad_labels = []

    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue

        path = os.path.join(label_dir, fname)
        with open(path, "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines, start=1):
            parts = line.strip().split()

            if len(parts) != 5:
                bad_labels.append((fname, i, "wrong number of fields"))
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                bad_labels.append((fname, i, "non-numeric value"))
                continue

            if cls not in valid_classes:
                bad_labels.append((fname, i, f"invalid class {cls}"))

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad_labels.append((fname, i, "bbox out of range"))

    if verbose:
        print("Bad label entries:", len(bad_labels))
        if len(bad_labels) > 0:
            print("Sample:", bad_labels[:10])

    return bad_labels

bad_labels = validate_yolo_labels(train_label_path)

In [ ]:
# Visualize some of the bad labels

plt.figure(figsize=(15, 15))

for i, (fname, line_no, reason) in enumerate(bad_labels[:13], start=1):
    label_path = os.path.join(train_label_path, fname)

    base = os.path.splitext(fname)[0]
    img_path = None
    for ext in [".jpg", ".png", ".jpeg"]:
        candidate = os.path.join(train_img_path, base + ext)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        print(f"Image not found for {fname}")
        continue

    img = plot_image_with_boxes(img_path, label_path)
    if img is None:
        continue

    plt.subplot(3, 5, i)
    plt.imshow(img)
    plt.title(f"{fname}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Clean up the bad bounding boxes:
# Clip the bounding box to between 0,1

bad_lookup = {(fname, line_no) for fname, line_no, _ in bad_labels}

for fname in os.listdir(train_label_path):
    if not fname.endswith(".txt"):
        continue

    path = os.path.join(train_label_path, fname)
    with open(path, "r") as f:
        lines = f.readlines()

    updated_lines = []

    for i, line in enumerate(lines, start=1):
        parts = line.strip().split()

        if len(parts) != 5:
            updated_lines.append(line)
            continue

        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            updated_lines.append(line)
            continue

        if (fname, i) in bad_lookup:
            x = min(max(x, 0.0), 1.0)
            y = min(max(y, 0.0), 1.0)
            w = min(max(w, 1e-6), 1.0)
            h = min(max(h, 1e-6), 1.0)
            updated_lines.append(f"{cls} {x} {y} {w} {h}\n")
        else:
            updated_lines.append(line)

    with open(path, "w") as f:
        f.writelines(updated_lines)

print("Clipped only the bad entries.")
bad_labels_after = validate_yolo_labels(train_label_path)

In [ ]:
from collections import Counter
import os

# Count the number of instances of each class in the training labels to understand class distribution 
# and identify any potential class imbalance issues that could affect model training
# it is useful to know if the imbalance will affect the model learning

counter = Counter()
bad_lines = []

for fname in os.listdir(train_label_path):
    if not fname.endswith(".txt"):
        continue
        
    path = os.path.join(train_label_path, fname)
    
    with open(path, "r") as f:
        for i, line in enumerate(f, start=1):
            parts = line.strip().split()
            
            if len(parts) < 1:
                bad_lines.append((fname, i, "empty line"))
                continue
            
            try:
                cls = int(parts[0])
                counter[cls] += 1
            except:
                bad_lines.append((fname, i, "invalid class value", line.strip()))

print("Class counts:", counter)
print("Bad lines found:", len(bad_lines))
print("Sample bad lines:", bad_lines[:10])

## Training Model


In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

# This cell is for training the model using CUDA.
# YOLOv11n is used to train for faster training time, however, the model may not be as accurate as larger
# model such as YOLOv11s or YOLOv11m

# Model is configured to train for 100 epochs with early stopping of 5 epochs if the validation does not improve.
# Mosaic and mixup augmentations are used with probabilities of 0.5 and 0.1 respectively.

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert Path("data.yaml").exists(), "data.yaml not found"
assert Path("data/train/images").exists(), "train images folder missing"
assert Path("data/val/images").exists(), "val images folder missing"

model = YOLO("yolo11n.pt")

model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    cache=True,
    project="runs/detect",
    name="yolo11_fire_smoke",
    patience=5,
    mosaic=0.5,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

## Evaluation

In [ ]:
# After training, we can evaluate the model on the validation set and get the model metrics 
# # such as mAP, precision, recall, and confusion matrices.

best_model = YOLO("runs/detect/yolo11_fire_smoke/weights/best.pt")
metrics = best_model.val(data="data.yaml",
                         split="val",
                         project=RUNS_ROOT / "detect",
                         name="val_metrics"
                         )
print(metrics)

In [ ]:
best_model.predict(
        source="data/val/images",
        conf=0.32,
        save=True,
        project=RUNS_ROOT / "detect",
        name="val_inference"
    )

### Validating Prediction

In [ ]:
# Comparing predictions with ground truth for a few samples to assess what the model is getting right and wrong, 
# which can help identify any common failure modes or areas for improvement in the model

def plot_predictions(img_path, results, conf_threshold=0.32):
    img = cv2.imread(img_path)

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        if conf < conf_threshold:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        color = (0, 255, 0) if cls == 0 else (0, 0, 255)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{'smoke' if cls == 0 else 'fire'} {conf:.2f}"
        cv2.putText(img, 
                    label, 
                    (x1, y1 - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, color, 2)
        
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    

In [ ]:
import random, os
from matplotlib import pyplot as plt

# visulize the ground truths and predictions side by side to understand the model performance

sample_imgs = random.sample(os.listdir("data/val/images"), 9)
# sample_imgs = os.listdir("data/val/images")[160:169]

plt.figure(figsize=(18,12))
for i, img_name in enumerate(sample_imgs):
    img_path = os.path.join("data/val/images", img_name)
    label_path = os.path.join("data/val/labels", img_name)
    gt_img = plot_image_with_boxes(img_path, label_path)
    results = best_model(img_path)
    pred_img = plot_predictions(img_path, results[0])
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor("#0e0e0e")
    fig.suptitle(img_name, color="#aaa", fontsize=9)

    axes[0].imshow(gt_img)
    axes[0].set_title(f"Ground Truth",
                      color="white", fontsize=12, pad=8)
    axes[0].axis("off")

    axes[1].imshow(pred_img)
    axes[1].set_title(f"Prediction",
                      color="white", fontsize=12, pad=8)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


## Inference

In [ ]:
# Test dataset is used to evaluate the final model performance on unseen data, 
# which gives us an estimate of how well the model will perform in real-world scenarios

best_model = YOLO("runs/detect/yolo11_fire_smoke/weights/best.pt")

test_metrics = best_model.val(data="data.yaml", 
                              split="test", 
                                project=RUNS_ROOT / "detect",
                                name="test_metrics")

results = best_model.predict(
    source="data/test/images",
    conf=0.32,
    save=True,
    project=RUNS_ROOT / "detect",
    name="final_test_inference"
)

### UAV Inference

In [ ]:
### Inference on UAV images to assess how well the model can be transferred to real-world data 
# and different environments.


best_model.predict(
    source="data/UAV_data/UAV_images",
    conf=0.32,
    save=True,
    project=RUNS_ROOT / "detect",
    name="uav_inference"
)

## Video Demo

In [ ]:
# Further assessment of the model performance on a UAV video to see how well it can perform in moving
# scenes and with different angles, lighting, and backgrounds compared to the static images.
import cv2
from ultralytics import YOLO

model = YOLO("runs/detect/yolo11_fire_smoke/weights/best.pt")

video_path = "13236230_3840_2160_24fps.mp4"
output_path = "output.mp4"
conf = 0.25

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"avc1")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.predict(frame, conf=conf, verbose=False)
    annotated_frame = results[0].plot()
    out.write(annotated_frame)

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Saved video to {output_path}")
